In [ ]:
!pip install faiss-cpu sentence-transformers langchain pypdf openai tiktoken

In [ ]:
!pip install -U langchain langchain-community pypdf

In [ ]:
!pip uninstall langchain -y
!pip install langchain langchain-community

Found existing installation: langchain 1.2.15
Uninstalling langchain-1.2.15:
  Successfully uninstalled langchain-1.2.15
  Using cached langchain-1.2.15-py3-none-any.whl.metadata (5.8 kB)
Using cached langchain-1.2.15-py3-none-any.whl (112 kB)


In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [ ]:
pdf_path = "sample_rag_dataset"  # change this to your file name

loader = PyPDFLoader("/content/sample_rag_dataset.pdf")
documents = loader.load()

print("Total pages loaded:", len(documents))
print(documents[0].page_content[:500])

Total pages loaded: 1
Title: Introduction to Artificial Intelligence and RAG
Artificial Intelligence (AI) is a field of computer science focused on building systems that can
perform tasks requiring human intelligence. These tasks include learning, reasoning,
problem-solving, and language understanding.
Machine Learning (ML) is a subset of AI that enables systems to learn from data without explicit
programming. Deep Learning is a further subset that uses neural networks with multiple layers.
Natural Language Processin


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))
print(chunks[0].page_content)

Total chunks: 5
Title: Introduction to Artificial Intelligence and RAG
Artificial Intelligence (AI) is a field of computer science focused on building systems that can
perform tasks requiring human intelligence. These tasks include learning, reasoning,
problem-solving, and language understanding.
Machine Learning (ML) is a subset of AI that enables systems to learn from data without explicit
programming. Deep Learning is a further subset that uses neural networks with multiple layers.


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [chunk.page_content for chunk in chunks]
embeddings = model.encode(texts)

print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding shape: (5, 384)


In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("Total vectors stored:", index.ntotal)

Total vectors stored: 5


In [ ]:
def search(query, top_k=3):
    query_vector = model.encode([query])

    distances, indices = index.search(np.array(query_vector), top_k)

    results = []
    for i in indices[0]:
        results.append(texts[i])

    return results

In [ ]:
query = "What is Retrieval-Augmented Generation?"

results = search(query)

for i, res in enumerate(results):
    print(f"\nResult {i+1}:\n", res)


Result 1:
 Large Language Models (LLMs), such as GPT, are built using transformer architectures. They are
trained on massive datasets and can generate human-like text.
However, LLMs suffer from a major issue called hallucination. This means they sometimes
generate incorrect or fabricated information confidently.
Retrieval-Augmented Generation (RAG) is a technique designed to reduce hallucination. It works
by retrieving relevant documents from a knowledge base and providing them as context to the

Result 2:
 language model before generating an answer.
The RAG pipeline consists of two main components: a retriever and a generator. The retriever
searches for relevant information, while the generator produces the final answer.
Despite its advantages, RAG has limitations. It may retrieve irrelevant documents if the query is not
well-formed. It also relies heavily on similarity search, which may not always capture true relevance.

Result 3:
 Improving RAG involves enhancing query understandi

In [ ]:
queries = [
    "What is AI?",
    "What is hallucination?",
    "Explain transformers",
    "What are applications of RAG?"
]

for q in queries:
    print("\n\nQuery:", q)
    results = search(q)
    for r in results:
        print("-", r[:150])



Query: What is AI?
- Title: Introduction to Artificial Intelligence and RAG
Artificial Intelligence (AI) is a field of computer science focused on building systems that ca
- Natural Language Processing (NLP) is an area of AI that focuses on interaction between computers
and humans through language. It is used in applicatio
- Large Language Models (LLMs), such as GPT, are built using transformer architectures. They are
trained on massive datasets and can generate human-like


Query: What is hallucination?
- Large Language Models (LLMs), such as GPT, are built using transformer architectures. They are
trained on massive datasets and can generate human-like
- Natural Language Processing (NLP) is an area of AI that focuses on interaction between computers
and humans through language. It is used in applicatio
- Title: Introduction to Artificial Intelligence and RAG
Artificial Intelligence (AI) is a field of computer science focused on building systems that ca


Query: Explain transformers

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_length=200
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCa

In [ ]:
!pip install -U transformers

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",   # ✅ supported in your version
    model="google/flan-t5-base",
    max_length=256,
    do_sample=False
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_length', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCau

In [ ]:
def generate_answer(query):
    retrieved_chunks = search(query)

    # Clean and limit context
    context = "\n".join(retrieved_chunks[:2])  # only top 2 chunks

    prompt = f"""
    Answer in 2-3 lines.

    Context:
    {context}

    Question:
    {query}
    """

    response = generator(prompt)

    return response[0]['generated_text']

In [ ]:
answer = generate_answer(query)

# Clean output
answer = answer.split("Answer:")[-1] if "Answer:" in answer else answer

print(answer)


    Answer in 2-3 lines.

    Context:
    critical applications.
Retrieval-Augmented Generation (RAG) is a hybrid approach that combines information retrieval
with text generation. Instead of relying solely on internal model knowledge, RAG retrieves relevant
documents from an external knowledge base.
The RAG architecture consists of two components: a retriever and a generator. The retriever
identifies relevant documents based on a query, while the generator uses this information to
produce a final response.
Context filtering involves selecting only the most relevant information from retrieved documents.
This reduces noise and helps the generator focus on important content.
In practical applications, RAG is used in question answering systems, search engines, and
conversational agents. It improves factual accuracy by grounding responses in external knowledge.
Evaluation of RAG systems can be qualitative or quantitative. Qualitative evaluation involves

    Question:
    What is RAG?
  

improvement


In [ ]:
def refine_query(query):
    # simple rule-based improvement (no API needed)

    if "RAG" in query:
        return "What is Retrieval-Augmented Generation in NLP and how does it work?"

    if "AI" in query:
        return "What is Artificial Intelligence and its applications?"

    return query

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def improved_search(query, top_k=3):
    query_vector = model.encode([query])

    distances, indices = index.search(np.array(query_vector), top_k)

    retrieved = [texts[i] for i in indices[0]]

    # Re-ranking
    retrieved_embeddings = model.encode(retrieved)

    scores = cosine_similarity(query_vector, retrieved_embeddings)[0]

    ranked = sorted(zip(retrieved, scores), key=lambda x: x[1], reverse=True)

    return [r[0] for r in ranked]

In [ ]:
def improved_generate_answer(query):

    refined_query = refine_query(query)

    retrieved_chunks = improved_search(refined_query)

    context = "\n".join(retrieved_chunks[:2])

    prompt = f"""
    Context:
    {context}

    Question: {refined_query}

    Give a short and clear answer in 2-3 lines:
    """

    response = generator(prompt)

    return response[0]['generated_text'].strip()

In [ ]:
query = "What is RAG?"

print("🔴 Baseline:")
print(generate_answer(query))

print("\n🟢 Improved:")
print(improved_generate_answer(query))

🔴 Baseline:

    Answer in 2-3 lines.

    Context:
    Improving RAG involves enhancing query understanding, re-ranking retrieved documents, and
filtering out low-quality information.
Applications of RAG include question answering systems, customer support bots, and
knowledge-based assistants.
language model before generating an answer.
The RAG pipeline consists of two main components: a retriever and a generator. The retriever
searches for relevant information, while the generator produces the final answer.
Despite its advantages, RAG has limitations. It may retrieve irrelevant documents if the query is not
well-formed. It also relies heavily on similarity search, which may not always capture true relevance.

    Question:
    What is RAG?
    

🟢 Improved:
Context:
    Large Language Models (LLMs), such as GPT, are built using transformer architectures. They are
trained on massive datasets and can generate human-like text.
However, LLMs suffer from a major issue called hallucination